Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [1]:
NO_RECORDS = 100

In [2]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, clean_doi, enrich_subjects_from_openaire, merge_subjects_from_new_method

d:\eosc_dc\metadata-warehouse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [4]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [5]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
ORDER BY id
OFFSET 400
LIMIT %s
;
"""

In [6]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/115093821', 'title': 'INTER Calibration cycle 18_3', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/115093821</identifier>\n    <datestamp>2021-09-14T09:12:03.246Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier>https://data.isis.stfc.ac.uk/browse/instrument/35/facilityCycle//investigation/115093821</identifier>\n          <alternate

In [7]:
record_titles

['INTER Calibration cycle 18_3',
 'INTER Calibration cycle 18_3',
 'INTER Calibration cycle 18_3',
 'INTER Calibration cycle 18_3',
 'Calibration',
 'Characterization of lipid nanoparticle',
 'Loki detector comissioning',
 'Various Sans2d calibration',
 'Front End upgrade commissioning',
 'Startup and Calibration',
 'Comissiioning',
 'IMAT calibration: metals',
 '3He setup calibration',
 'INTER Calibration cycle 18_3',
 'Detector Calibration',
 'New beam maps of ChipIR',
 'IMAT calibration: powders',
 'Startup',
 'Commissioned of IMAT MCP after ovehaul',
 'Probing Hydrogen evolution detection for corrosion studies',
 'IMAT calibration: powders',
 'IBEX commissionning',
 'Imperial Training Course',
 'Spin-Echo Commissioning',
 'HTS110 Magnet Pol Testing',
 'Polarisation calibration',
 'INTER Calibration cycle 18_3',
 'Instrument calibration',
 'Startup',
 'Imperial Training Course (Day 2)',
 'new snout commissioning',
 'Scattering contrast of single magnetic nanoflowers',
 'Express day 

In [8]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/115093829</identifier>\n    <datestamp>2021-09-14T09:12:03.582Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier>https://data.isis.stfc.ac.uk/browse/instrument/35/facilityCycle//investigation/115093829</identifier>\n          <alternateIdentifiers alternateIdentifierType="RB">\n            <alternateIdentifier>1835571</alternateIdentifier>\n          </a

In [9]:
get_doi(record_raw_metadata[2])

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [10]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

Old methods, DOI-by-DOI. outdated

In [11]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

NO DOI


In [12]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[5, 45, 70]


Batch methods

In [13]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/115095273', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116344490', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116731821']
['10.5286/isis.e.rb1890206-1', '10.5286/isis.e.rb2230015-1', '10.5286/isis.e.rb2230017-1']


In [14]:
print(NO_RECORDS)
print(len(doi_to_enrich))

100
3


In [15]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, connect_ids_with_subjects_by_doi, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'Dr Ron Smith', 'name': 'Ron', 'surname': 'Smith', 'rank': 1, 'pid': None}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'und', 'label': 'Undetermined'}, 'countries': None, 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'STFC ISIS Neutron and Muon Data'}, 'provenance': None}], 'mainTitle': 'Certification of NIST SRMs 676b, 674c and 1878c', 'subTitle': None, 'descriptions': None, 'publicationDate': '2022-01-01', 'publisher': 'ISIS Neutron and Muon Source', 'embargoEndDate': None, 'sources': None, 'formats': None, 'contributors': None, 'coverages': None, 'bestAccessRight': {'code': 'c_abf2', 'label': 'OPEN', 'scheme': 'http://vocabularies.coar-repositories.org/documentation/access_rights/'}, 'container': None, 'documentationUrls': None, 'codeRepositoryUrl': None, 'programmingLanguage': None, 'contactPeople': None, 'contactGroups': None, 'tools': None, 'size': None, 'version': None, 'geoLocations': None, 'id'

In [16]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.5286/isis.e.rb2230017-1',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb2230017',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1890206',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1890206-1',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb2230015',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb2230015-1',
  'subjects': ['STFC ISIS Neutron and Muon Data']}]

In [17]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/115095273',
  'doi': '10.5286/isis.e.rb1890206-1'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116344490',
  'doi': '10.5286/isis.e.rb2230015-1'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116731821',
  'doi': '10.5286/isis.e.rb2230017-1'}]

In [18]:
enriched_list = connect_ids_with_subjects_by_doi(id_dois, doi_subjects)
enriched_list

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/115095273',
  'doi': '10.5286/isis.e.rb1890206-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116344490',
  'doi': '10.5286/isis.e.rb2230015-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116731821',
  'doi': '10.5286/isis.e.rb2230017-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}}]

In [19]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects']]

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/115095273',
  'doi': '10.5286/isis.e.rb1890206-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116344490',
  'doi': '10.5286/isis.e.rb2230015-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116731821',
  'doi': '10.5286/isis.e.rb2230017-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}}]

In [20]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/115095273',
  'doi': '10.5286/isis.e.rb1890206-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116344490',
  'doi': '10.5286/isis.e.rb2230015-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116731821',
  'doi': '10.5286/isis.e.rb2230017-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}}]

In [21]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects']]))

3
3


In [22]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects']]

first_enriched = full_info_enriched[:10]

In [23]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[5, 45, 70]

In [24]:
first_enriched

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/115095273',
  'doi': '10.5286/isis.e.rb1890206-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116344490',
  'doi': '10.5286/isis.e.rb2230015-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116731821',
  'doi': '10.5286/isis.e.rb2230017-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}}]

In [25]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/115095273</identifier>\n    <datestamp>2021-08-24T13:14:00.577Z</datestamp>\n    <setSpec>doiInvs</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier identifierType="DOI">https://doi.org/10.5286/ISIS.E.RB1890206-1</identifier>\n          <alternateIdentifiers alternateIdentifierType="RB">\n            <alternateIdentifier>1890206</alternateIdentifier>\n     

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [26]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [27]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [28]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.1056/NEJMoa2034577',
    '10.1056/NEJMoa2021436',
    '10.1056/NEJMoa1511939',
    '10.1056/NEJMoa1800389'
]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

In [29]:
from utils.enrichment_methods import batch_enrich_subjects_from_openalex

enriched_results = batch_enrich_subjects_from_openalex(
    id_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results = [el for el in enriched_results if el['enriched_subjects']]
print(len(enriched_results))
enriched_results

0


[]

In [30]:
# Showcase  - manually input DOIs

test_dois = [
    {'id': "testid1",
     'doi': '10.1016/j.paid.2021.110888'},
    {'id': "testid2",
     'doi': '10.23662/fors-ds-686-1'},
    {'id': "testid3",
     'doi': '10.34894/YGHDXB'},
    {'id': "testid3",
     'doi': '10.15151/esrf-es-1580558984'}
]

enriched_results = batch_enrich_subjects_from_openalex(
    test_dois,
    OpenAlex_API_key,
    score_treshold = 0.7
)

enriched_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': set()},
 {'id': 'testid3', 'doi': '10.34894/YGHDXB', 'enriched_subjects': set()},
 {'id': 'testid3',
  'doi': '10.15151/esrf-es-1580558984',
  'enriched_subjects': set()}]

In [31]:
# Compare openalex with openaire:
oaire_results = batch_enrich_subjects_from_openaire(test_dois)
oaire_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}},
 {'id': 'testid3',
  'doi': '10.34894/YGHDXB',
  'enriched_subjects': {'Arts and Humanities',
   'Earth and Environmental Science',
   'Earth and Environmental Sciences',
   'Environmental Research',
   'Geosciences',
   'Humanities',
   'Natural Sciences'}},
 {'id': 'testid3',
  'doi': '10.15151/esrf-es-1580558984',
  'enriched_subjects': {'BM29',
   'MX-2590',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}}]

In [32]:
merge_subjects_from_new_method(enriched_results, oaire_results)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality',
   'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}},
 {'id': 'testid3',
  'doi': '10.34894/YGHDXB',
  'enriched_subjects': {'BM29',
   'MX-2590',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': 'testid3',
  'doi': '10.15151/esrf-es-1580558984',
  'enriched_subjects': {'BM29',
   'MX-2590',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}}]

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

In [33]:
## DATACITE REST API

DATACITE_API_URL = "https://api.datacite.org/dois"

DataCite provides an additional "enriched" option for fetching doi-by-doi, could be interesting.

In [34]:
from utils.enrichment_methods import batch_enrich_subjects_from_datacite, _datacite_batch_doi_resolve, _datacite_get_doi_subjects

In [35]:
DC_res = _datacite_batch_doi_resolve(doi_to_enrich)

In [36]:
DC_res

[{'id': '10.5286/isis.e.rb1890206-1',
  'type': 'dois',
  'attributes': {'doi': '10.5286/isis.e.rb1890206-1',
   'prefix': '10.5286',
   'suffix': 'isis.e.rb1890206-1',
   'identifiers': [],
   'alternateIdentifiers': [],
   'creators': [{'name': 'Mr Weimin Li',
     'nameType': 'Personal',
     'affiliation': [],
     'nameIdentifiers': []}],
   'titles': [{'title': 'Characterization of lipid nanoparticle'}],
   'publisher': 'ISIS Facility',
   'container': {'type': 'DataRepository',
    'identifier': '10.5286/ISIS.E.RB1890206',
    'identifierType': 'DOI'},
   'publicationYear': 2018,
   'subjects': [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
   'contributors': [],
   'dates': [{'date': '2018', 'dateType': 'Issued'}],
   'language': None,
   'types': {'ris': 'DATA',
    'bibtex': 'misc',
    'citeproc': 'dataset',
    'schemaOrg': 'Dataset',
    'resourceTypeGeneral': 'Dataset'},
   'relatedIdentifiers': [{'relationType': 'IsPartOf',
     'relatedIdentifier': '10.

In [37]:
[el['attributes']['subjects'] for el in DC_res]

[[{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
 [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
 [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}]]

In [38]:
slist = [el['attributes']['subjects'] for el in DC_res]

[[el.get('subject') for el in sublist] for sublist in slist]

[['STFC ISIS Neutron and Muon Data'],
 ['STFC ISIS Neutron and Muon Data'],
 ['STFC ISIS Neutron and Muon Data']]

In [39]:
_datacite_get_doi_subjects(DC_res)

[{'doi': '10.5286/isis.e.rb1890206-1',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb2230015-1',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb2230017-1',
  'subjects': ['STFC ISIS Neutron and Muon Data']}]

In [40]:
DC_enriched = batch_enrich_subjects_from_datacite(id_dois)

In [41]:
[el for el in DC_enriched if el['enriched_subjects']]

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/115095273',
  'doi': '10.5286/isis.e.rb1890206-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116344490',
  'doi': '10.5286/isis.e.rb2230015-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/116731821',
  'doi': '10.5286/isis.e.rb2230017-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}}]

In [42]:
len([el for el in DC_enriched if el['enriched_subjects']])

3

In [43]:
# All in one method:
from utils.enrichment_methods import batch_enrich_subjects_from_datacite
batch_enrich_subjects_from_datacite(test_dois)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': set()},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': set()},
 {'id': 'testid3',
  'doi': '10.34894/YGHDXB',
  'enriched_subjects': {'Arts and Humanities',
   'Earth and Environmental Sciences'}},
 {'id': 'testid3',
  'doi': '10.15151/esrf-es-1580558984',
  'enriched_subjects': {'BM29',
   'MX-2590',
   'Macromolecular Crystallography',
   'x-ray probe'}}]

# Related DOIs  
Extract related identifiers that are also DOIs. If they are in the database, use their subjects. If not, use DOI-resolving methods on them.

Note:
a DOI must start with 10. prefix. I've seen some related DOIs that start with 0.  , if I add a 1 to the start it resolves properly.

## SQL query to see records with related DOIs   
```
--- get records with related identifiers of any tipe
WITH query AS (
    SELECT
        raw_metadata,
		doi as record_doi,
        xpath($$(//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI'])/text()$$, raw_metadata)::text[] AS related
    FROM public.records
)
SELECT *
FROM query
WHERE array_length(related, 1) > 0
LIMIT 1000;
```

47k records have SOME relatedIdentifiers - lots of them ISSN or proprietary ids

In [44]:
# Let's try out some records with related DOIs, for testing we won't limit ourselves to records with no raw_subjects.
# This will get 100 records with some related DOIs

query_related_dois = """
WITH query AS (
    SELECT
        id,
        raw_metadata,
		doi as record_doi,
        ARRAY(
            SELECT jsonb_array_elements(datacite_json->'subjects')->>'subject'
        ) AS subjects,
        xpath($$(//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI'])/text()$$, raw_metadata)::text[] AS related
    FROM public.records
)
SELECT *
FROM query
WHERE array_length(related, 1) > 0
LIMIT 100;
"""

In [45]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_related_dois)
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_subjects = [row["subjects"] for row in results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]
        record_related_dois = [row["related"] for row in results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2591', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:rodare.hzdr.de:2591</identifier>\n    <datestamp>2023-12-08T09:29:38Z</datestamp>\n    <setSpec>openaire_data</setSpec>\n    <setSpec>user-rodare</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.0/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.0/ oai_datacite.xsd">\n      <isReferenceQuality>true</isReferenceQuality>\n      <schemaVersion>4.1</schemaVersion>\n      <datacentreSymbol>CERN.ZENODO</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n          <identifier identifierType="DOI">10.14278/rodare.2591</identifier>\n          <creators>\n            <

In [46]:
results

[{'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::2591',
  'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:rodare.hzdr.de:2591</identifier>\n    <datestamp>2023-12-08T09:29:38Z</datestamp>\n    <setSpec>openaire_data</setSpec>\n    <setSpec>user-rodare</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.0/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.0/ oai_datacite.xsd">\n      <isReferenceQuality>true</isReferenceQuality>\n      <schemaVersion>4.1</schemaVersion>\n      <datacentreSymbol>CERN.ZENODO</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n          <identifier identifierType="DOI">10.14278/rodare.2591</identifier>\n          <creators>\n           

In [47]:
# For a specific DOI set, search our datasetdb and retrieve subjects if any.
# Maybe a record references/is part of another dataset so the subjects are relevant too.

# This query works for a list of DOIs -  if a record has multiples

query_subjects_by_doi = """
WITH query AS (
    SELECT
    id,
        ARRAY(
            SELECT jsonb_array_elements(datacite_json->'subjects')->>'subject'
        ) AS subjects
    FROM public.records
)
SELECT subjects
FROM query
WHERE query.id = ANY(%s)
LIMIT 100;
"""

In [48]:
# Some related dois link to just https://doi.org/

# We'll first apply the cleaning, then drop the empty strings

record_related_dois = [[clean_doi(doi) for doi in sublist if clean_doi(doi)] for sublist in record_related_dois]



In [49]:
record_related_dois

[['10.14278/rodare.2590'],
 ['10.14278/rodare.4121'],
 ['10.14278/rodare.1755'],
 ['10.14278/rodare.269'],
 ['10.14278/rodare.3551'],
 ['10.14278/rodare.802'],
 ['10.14278/rodare.76', '10.1115/1.4045497', '10.14278/rodare.212'],
 ['10.14278/rodare.3391'],
 ['10.1063/5.0135729', '10.14278/rodare.2163'],
 ['10.17815/jlsrf-3-159', '10.14278/rodare.3938'],
 ['10.14278/rodare.1374'],
 ['10.17815/jlsrf-3-159', '10.14278/rodare.4104'],
 ['10.14278/rodare.3745'],
 ['10.1103/physrevb.104.184429', '10.14278/rodare.1277'],
 ['10.14278/rodare.4447'],
 ['10.14278/rodare.357'],
 ['10.14278/rodare.1693'],
 ['10.14278/rodare.4123'],
 ['10.17815/jlsrf-3-159', '10.14278/rodare.3403'],
 ['10.1103/physrevresearch.5.013049', '10.14278/rodare.1998'],
 ['10.1021/acs.jmedchem.5c00890', '10.14278/rodare.3890'],
 ['10.58065/24020',
  '10.17815/jlsrf-2-58',
  '10.1016/j.nima.2022.166701',
  '10.1016/j.nima.2018.09.062',
  '10.1016/j.sna.2023.114859',
  '10.1109/trpms.2019.2930362',
  '10.1088/1361-6560/ab176d',


In [50]:
first_related_doi = record_related_dois[0]
first_related_doi

['10.14278/rodare.2590']

In [51]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_subjects_by_doi, (first_related_doi,))
        
        record_res = cur.fetchall()
except Exception as e:
    print(f"Query error: {e}")
        

record_res

[]

In [52]:
# Go through all test cases

related_subjects = []

for i in range(len(record_ids)):
    try:
        with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
            cur = conn.cursor()
            related_dois = record_related_dois[i]

            # If there are no related DOIs remaining after cleaning, no subjects will be enriched
            if not related_dois:
                related_subjects.append(set())
                continue
            
            cur.execute(query_subjects_by_doi, (related_dois,))
            
            record_res = cur.fetchall()
            related_subjects.append(record_res)
    except:
        print(f"Query error: {e}")
        

In [53]:
# No hits
[res for res in related_subjects if res]

[]

In [54]:
# Extra: go through our DB
# and ALSO retrieve all subjects from openaire and openalex for those individual DOIs

related_subjects = []

for i in range(len(record_ids)):
    try:
        with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
            cur = conn.cursor()
            related_dois = record_related_dois[i]

            if not related_dois:
                related_subjects.append(set())
                continue

            cur.execute(query_subjects_by_doi, (related_dois,))
            
            record_res = cur.fetchall()
            record_res = set(record_res)
            # We need an id-doi pairing for each doi, we'll make fake ids since we won't use them for joining later
            id_dois = [{'id': str(i), 'doi': doi,} for i, doi in enumerate(related_dois)]
            if (i < 10):
                print(id_dois)

            openaire = batch_enrich_subjects_from_openaire(id_dois)
            openaire_subjects = set().union(*(el["enriched_subjects"] for el in openaire))
            openaire_subjects = {"OpenAIRE: " + el for el in openaire_subjects}

            openalex = batch_enrich_subjects_from_openalex(id_dois, OpenAlex_API_key)
            openalex_subjects = set().union(*(el["enriched_subjects"] for el in openalex))
            openalex_subjects = {"OpenAlex: " + el for el in openalex_subjects}

            datacite = batch_enrich_subjects_from_datacite(id_dois)
            datacite_subjects = set().union(*(el["enriched_subjects"] for el in datacite))
            datacite_subjects = {"Datacite: " + el for el in datacite_subjects}

            # Union of all methods
            record_res = record_res | openaire_subjects | openalex_subjects | datacite_subjects

            related_subjects.append(record_res)
    except Exception as e:
        print(f"Query error: {e}")
        

[{'id': '0', 'doi': '10.14278/rodare.2590'}]
[{'id': '0', 'doi': '10.14278/rodare.4121'}]
[{'id': '0', 'doi': '10.14278/rodare.1755'}]
[{'id': '0', 'doi': '10.14278/rodare.269'}]
[{'id': '0', 'doi': '10.14278/rodare.3551'}]
[{'id': '0', 'doi': '10.14278/rodare.802'}]
[{'id': '0', 'doi': '10.14278/rodare.76'}, {'id': '1', 'doi': '10.1115/1.4045497'}, {'id': '2', 'doi': '10.14278/rodare.212'}]
[{'id': '0', 'doi': '10.14278/rodare.3391'}]
[{'id': '0', 'doi': '10.1063/5.0135729'}, {'id': '1', 'doi': '10.14278/rodare.2163'}]
[{'id': '0', 'doi': '10.17815/jlsrf-3-159'}, {'id': '1', 'doi': '10.14278/rodare.3938'}]


In [55]:
related_subjects

[{'Datacite: Actinides',
  'Datacite: Lanthanide-binding proteins',
  'Datacite: Lanthanide-dependent bacteria',
  'Datacite: Lanthanides',
  'Datacite: Methane',
  'Datacite: Methanol dehydrogenases',
  'OpenAIRE: Actinides',
  'OpenAIRE: Engineering Sciences',
  'OpenAIRE: Lanthanide binding proteins',
  'OpenAIRE: Lanthanide dependent bacteria',
  'OpenAIRE: Lanthanide-binding proteins',
  'OpenAIRE: Lanthanide-dependent bacteria',
  'OpenAIRE: Lanthanides',
  'OpenAIRE: Life Sciences',
  'OpenAIRE: Methane',
  'OpenAIRE: Methanol dehydrogenases',
  'OpenAIRE: Natural Sciences'},
 {'Datacite: Code Generation',
  'Datacite: Directed Acyclic Graph',
  'Datacite: HPC',
  'Datacite: Heterogeneity',
  'Datacite: Julia',
  'Datacite: Optimization',
  'Datacite: QED',
  'OpenAIRE: Code Generation',
  'OpenAIRE: Directed Acyclic Graph',
  'OpenAIRE: HPC',
  'OpenAIRE: Heterogeneity',
  'OpenAIRE: Julia',
  'OpenAIRE: Optimization',
  'OpenAIRE: QED',
  'OpenAlex: Computation'},
 {'Datacite: